In [3]:
!pip install -r requirements.txt

  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached pyarrow-23.0.1-cp314-cp314-macosx_12_0_x86_64.whl.metadata (3.1 kB)
  Using cached matplotlib-3.10.8-cp314-cp314-macosx_10_13_x86_64.whl.metadata (52 kB)
  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached contourpy-1.3.3-cp314-cp314-macosx_10_13_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.62.1-cp314-cp314-macosx_10_15_x86_64.whl.metadata (117 kB)
  Using cached kiwisolver-1.5.0-cp314-cp314-macosx_10_15_x86_64.whl.metadata (5.1 kB)
  Using cached packaging-26.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
Using cached beautifulsoup4-4.14.3-py3-n

In [1]:
from pathlib import Path
import os

print("cwd:", Path.cwd())
print("existe notebooks?", Path("notebooks").exists())
print("arquivos aqui:", list(Path(".").iterdir())[:20])


cwd: /Users/felipeignacio/Projects/datajud_probe
existe notebooks? True
arquivos aqui: [PosixPath('.DS_Store'), PosixPath('requirements.txt'), PosixPath('CODEX_PROMPT.md'), PosixPath('README.md'), PosixPath('.gitignore'), PosixPath('teste.ipynb'), PosixPath('.env'), PosixPath('.venv'), PosixPath('.env.example'), PosixPath('.git'), PosixPath('data'), PosixPath('notebooks'), PosixPath('.sixth'), PosixPath('.idea'), PosixPath('src')]


In [3]:

from pathlib import Path
import pandas as pd
from bs4 import BeautifulSoup

path = Path("notebooks/78_Tabela_Assuntos_Justica_Federal_1_Grau.xls")

html = path.read_text(encoding="latin-1", errors="replace")
soup = BeautifulSoup(html, "html.parser")

main_table = soup.find("table")
rows = []

for tr in main_table.find_all("tr", recursive=False):
    cells = tr.find_all("td", recursive=False)

    if len(cells) < 6:
        continue

    texts = [cell.get_text(" ", strip=True) for cell in cells]

    assunto_cols = texts[:5]
    assunto = next((x for x in assunto_cols if x.strip()), None)

    if not assunto:
        continue

    rows.append({
        "assunto": assunto,
        "codigo": texts[5] if len(texts) > 5 else None,
        "codigo_pai": texts[6] if len(texts) > 6 else None,
        "dispositivo_legal": texts[7] if len(texts) > 7 else None,
        "artigo": texts[8] if len(texts) > 8 else None,
        "alteracoes": texts[9] if len(texts) > 9 else None,
        "glossario": texts[10] if len(texts) > 10 else None,
        "ods": texts[11] if len(texts) > 11 else None,
        "data_publicacao": texts[12] if len(texts) > 12 else None,
        "data_alteracao": texts[13] if len(texts) > 13 else None,
        "data_inativacao": texts[14] if len(texts) > 14 else None,
        "data_reativacao": texts[15] if len(texts) > 15 else None,
        "nivel_visual": next((i + 1 for i, x in enumerate(assunto_cols) if x.strip()), None),
        "fonte": path.name,
        "instancia": "Justiça Federal 1º Grau",
    })

df = pd.DataFrame(rows)

def normalize_codigo(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if not text:
        return None
    if text.endswith(".0"):
        text = text[:-2]
    if text.isdigit():
        return text.zfill(5)
    return text

df["codigo"] = df["codigo"].map(normalize_codigo)
df["codigo_pai"] = df["codigo_pai"].map(normalize_codigo)

print(df.shape)
print(df.head(20).to_string(index=False))
print("\nCódigo 03608:")
print(df[df["codigo"].eq("03608")].to_string(index=False))



(2418, 15)
                                            assunto codigo codigo_pai dispositivo_legal                                         artigo alteracoes                                                                                                                                                                                                                                                                         glossario                                                                          ods     data_publicacao      data_alteracao data_inativacao data_reativacao  nivel_visual                                         fonte               instancia
                                             Acesso  12795      12775           CF; LDB                                                                                                                                                                                                                                                         

In [1]:
import sys
print(sys.executable)


/Users/felipeignacio/Projects/datajud_probe/.venv/bin/python


In [ ]:
import sys
import pandas as pd
from bs4 import BeautifulSoup

print(sys.executable)
print("pandas", pd.__version__)
print("bs4 ok")
